# Deliverable 3: Our First Model, Classification with KNN

**MOD10 Introduction to Machine Learning**, November 2026

- **Group:** CUPILLARD Eliot, DENOUAL Romain, ROY-NOUGUIER Erwann
- **Dataset:** Arbres publics sur le territoire de la Ville (consolidated file), City of Montreal, licence CC BY 4.0, same unchanged copy as Deliverable 2 (https://doi.org/10.5281/zenodo.23540888)
- **Category (`y`):** trunk size class (young, medium, mature) built from `DHP`, the trunk diameter at 1.4 m
- **Question:** Can KNN tell the size class of a public tree from its species, location and age?

## 0. Setup and data from Deliverable 2

We reuse the data saved at the end of Deliverable 2 (`data/processed/trees_encoded.csv`): cleaned and encoded, not scaled. We do not clean or encode again. The file comes from the `to_csv` line of section 3 of that notebook and is kept in the repository.

`X` is the table without `DHP`.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

SEED = 42
DATA_PATH = Path("../data/processed/trees_encoded.csv")

assert DATA_PATH.is_file(), "File missing: run Deliverable 2 with the to_csv line of section 3 not commented"
data = pd.read_csv(DATA_PATH, index_col="raw_row")
assert data.shape == (334950, 62), "Wrong shape: this is not the file of Deliverable 2"

X = data.drop(columns=["DHP"])
print("Data:", data.shape, "| Missing values:", data.isna().sum().sum())

Data: (334950, 62) | Missing values: 0


## 1. Our category

Our target `DHP` is a number and KNN needs a category, so we build one from `DHP`: the size class of the trunk.

| Class | DHP |
|---|---|
| `young` | up to 10 |
| `medium` | above 10, up to 30 |
| `mature` | above 30 |

- The cut points are round values close to the thirds of the data (7 and 32). The choice is ours, not an official scale.
- A DHP of exactly 10 is `young`, exactly 30 is `medium` (6,107 and 4,777 trees).
- `DHP` is a diameter, not an age: `mature` means a large trunk.
- 70% of the `young` trees have a DHP of exactly 5, probably a default value (see Deliverable 2).

In [2]:
y = pd.cut(data["DHP"], bins=[0, 10, 30, 250], labels=["young", "medium", "mature"])
data["DHP"].groupby(y, observed=True).agg(["count", "min", "median", "max"])

,count,min,median,max
DHP,,,,
young,129358,1.0,5.0,10.0
medium,89385,10.1,21.0,30.0
mature,116207,30.1,47.0,246.0


## 2. Features and label

`y` is the size class and `X` the encoded table without `DHP`: the category comes from it, so keeping it would be data leakage. The identifiers and the columns that repeat other information are not in `X` (see Deliverable 2).

The classes are fairly balanced. A model that always answers `young` is right 38.6% of the time: this is our baseline.

In [3]:
print("X:", X.shape, "| y:", y.shape)
print("Columns that are not numbers:", list(X.select_dtypes(exclude="number").columns))
print("Missing values:", X.isna().sum().sum())
pd.DataFrame({"trees": y.value_counts(), "share": y.value_counts(normalize=True).round(3)}).sort_index()

X: (334950, 61) | y: (334950,)
Columns that are not numbers: []
Missing values: 0


,trees,share
DHP,,
young,129358,0.386
medium,89385,0.267
mature,116207,0.347


## 3. Split and scale

We reuse the split of Deliverable 2 (`test_size=0.2`, `random_state=SEED`, no `stratify`), so train and test have the same rows. The class shares are the same in both (below). Deliverable 2 saved the data before the split, so we split and scale again here.

Because KNN uses distances, we standardize the same three columns as in Deliverable 2 (`Latitude`, `Longitude`, `age`), with the scaler fitted on the training set only.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Class shares in train:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Class shares in test:", y_test.value_counts(normalize=True).round(3).to_dict())

Train: (267960, 61) | Test: (66990, 61)
Class shares in train: {'young': 0.386, 'mature': 0.347, 'medium': 0.267}
Class shares in test: {'young': 0.386, 'mature': 0.347, 'medium': 0.267}


In [5]:
columns_to_scale = ["Latitude", "Longitude", "age"]
scaler = StandardScaler()
# fit on the training set only
X_train[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])
X_test[columns_to_scale] = scaler.transform(X_test[columns_to_scale])

print("Mean and standard deviation after scaling:")
display(X_train[columns_to_scale].agg(["mean", "std"]).round(3))
display(X_test[columns_to_scale].agg(["mean", "std"]).round(3))

Mean and standard deviation after scaling:


,Latitude,Longitude,age
mean,0.0,-0.0,-0.0
std,1.0,1.0,1.0


,Latitude,Longitude,age
mean,0.003,0.003,-0.004
std,0.998,0.998,0.994


## 4. Train KNN and choose k

We compare the values of k on a validation set (20% of the training set), not on the test set as in class: if we kept the k with the best test score, the test score would be too optimistic. The test set is only used at the end.

The course suggests starting near the square root of the number of training rows (about 517). We try values from 1 to 517, at about a minute each. The scaler was fitted on the whole training set, validation rows included.

In [6]:
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=SEED)
print("Fit:", X_fit.shape, "| Validation:", X_val.shape)

for k in [1, 5, 9, 15, 51, 151, 517]:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_fit, y_fit)
    acc = accuracy_score(y_val, model.predict(X_val))
    print(f"k={k} | accuracy={acc:.3f}")

Fit: (214368, 61) | Validation: (53592, 61)


k=1 | accuracy=0.827


k=5 | accuracy=0.827


k=9 | accuracy=0.821


k=15 | accuracy=0.812


k=51 | accuracy=0.785


k=151 | accuracy=0.766


k=517 | accuracy=0.749


| k | 1 | 5 | 9 | 15 | 51 | 151 | 517 |
|---|---|---|---|---|---|---|---|
| Accuracy on the validation set | 0.827 | 0.827 | 0.821 | 0.812 | 0.785 | 0.766 | 0.749 |

We keep k = 5. The accuracy is the same for k = 1 and k = 5, then it goes down. We prefer k = 5 because with k = 1 the answer depends on a single tree. The best k is small: the closest trees are the most useful.

## 5. Evaluation

We train on the whole training set with k = 5 and test once on the test set. The rows of the matrix are the real classes, the columns the predicted ones. The baseline always answers `young`.

In [7]:
best_k = 5
model = KNeighborsClassifier(n_neighbors=best_k)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print(f"accuracy: {accuracy_score(y_test, preds):.3f}")
print(f"baseline (always young): {(y_test == 'young').mean():.3f}")

labels = ["young", "medium", "mature"]
print("confusion matrix (rows = real class, columns = predicted class):")
print(pd.DataFrame(confusion_matrix(y_test, preds, labels=labels), index=labels, columns=labels))

accuracy: 0.837
baseline (always young): 0.386
confusion matrix (rows = real class, columns = predicted class):
        young  medium  mature
young   23593    1760     514
medium   1602   12590    3705
mature    235    3116   19875


## 6. Interpretation

The accuracy on the test set is 0.837, against 0.386 for the baseline.

10,932 trees out of 66,990 are wrong (16%). Most mistakes are between neighbouring classes (10,183, or 93%). Only 749 trees (1%) go from `young` to `mature` or the other way.

`medium` is the weakest class: 70% of its trees are right, against 91% for `young` and 86% for `mature`. It is mixed with both neighbours, and the cut points are our choice.

We do not know which mistake costs more to the city. If the size is used to plan work on large trees, the worst one is a `mature` tree predicted smaller: 3,351 of 23,226 (14%).

The classes are not rare, so accuracy is a fair first number, but it hides the weakness of `medium`. The confusion matrix shows it.

The score is probably lower on a new area, because the random split puts trees of the same street in both train and test. We did not test this.

## 7. Model fit

KNN reaches 0.84 against 0.39 for the baseline, but it is slow on 335,000 trees and our real target is a number, so a regression model is probably a better fit (see "Choosing your model" in the Class 4 slides).